<a href="https://colab.research.google.com/github/mpodmanicky/cpis/blob/main/week_3/Task_2_backprop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 2
Implement basic backward pass using only numpy:
 - for your last week's Single Layer Perceptron.
 - for all activation functions
 - loss functions

Perform forward pass and backward pass, and use the gradient check function to verify your implementation...

In [2]:
import numpy as np
from utils import Module

## Linear Layer

In your previous task you defined `forward(input)` pass for your Linear class. Now we continue in creation of your own framework a little further with defining the `backward(dNet)` function. In this little framework is activation and linear unit separated. This separation is benefit in backward propagation and optimization. (If you want to know why, take a look on implementation of forward and backward propagation in class Model.)

Note, that now you are implementing backward pass for on only one sample, the lecture's framework was prepared for training on the whole dataset of $m$ samples.

In [22]:
#------------------------------------------------------------------------------
#   Linear class
#------------------------------------------------------------------------------
class Linear(Module):
    def __init__(self, in_features, out_features):
        super(Linear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.W = np.random.randn(out_features, in_features)
        self.dW = np.zeros_like(self.W)
        self.b = np.zeros((out_features, 1)) # Watch-out for the shape
        self.db = np.zeros_like(self.b)

    def forward(self, input: np.ndarray) -> np.ndarray:
        self.fw_inputs = input
        self.m = self.fw_inputs.shape[1]
        net = np.matmul(self.W, input) + self.b
        return net

    def backward(self, dz: np.ndarray) -> np.ndarray:
        # >>>>>>>>> add here
        dx = np.dot(self.W.T, dz)
        self.dW = np.dot(dz, self.fw_inputs.T)
        self.db = np.sum(dz, axis=1, keepdims=True)
        return dx
        # <<<<<<<<<

## Activations
Implement backward pass for Sigmoid, Tanh and ReLU activation functions.

In [4]:
#------------------------------------------------------------------------------
#   SigmoidActivationFunction class
#------------------------------------------------------------------------------
class Sigmoid(Module):
    def __init__(self):
        super(Sigmoid, self).__init__()

    def forward(self, input: np.ndarray) -> np.ndarray:
        self.fw_input = input
        return 1.0 / (1.0 + np.exp(-input))

    def backward(self, da) -> np.ndarray:
        # >>>>>>>>> add here
        s = 1.0 / (1.0 + np.exp(-self.fw_input))
        dx = da * s * (1 - s)
        return dx
        # <<<<<<<<<

#------------------------------------------------------------------------------
#   HyperbolicTangentActivationFunction class
#------------------------------------------------------------------------------
class Tanh(Module):
    def __init__(self):
        super(Tanh, self).__init__()

    def forward(self, input: np.ndarray) -> np.ndarray:
        self.fw_input = input
        return (np.exp(2 * input) - 1) / (np.exp(2 * input) + 1)

    def backward(self, da) -> np.ndarray:
        # >>>>>>>>> add here
        tanh_output = (np.exp(2 * self.fw_input) - 1) / (np.exp(2 * self.fw_input) + 1)
        dx = da * (1 - np.square(tanh_output))
        return dx
        # <<<<<<<<<

#------------------------------------------------------------------------------
#   RELUActivationFunction class
#------------------------------------------------------------------------------
class ReLU(Module):
    def __init__(self):
        super(ReLU, self).__init__()

    def forward(self, input: np.ndarray) -> np.ndarray:
        self.fw_input = input
        return np.maximum(input, 0)

    def backward(self, da) -> np.ndarray:
        # >>>>>>>>> add here
        dx = da * (self.fw_input > 0).astype(float)
        return dx
        # <<<<<<<<<


## Loss functions
For successful backward pass, the computation and derivation of Loss function is necessary.
The most common Loss functions are **Mean Square Error** _(MSE, L2)_ **Mean Absolute Error** _(MAE, L1)_ and **Binary Cross Entropy** _(BCE, Log Loss)_ and their modifications according to what is better for the current dataset.

Implement SE and BCE Loss functions as Modules of our little framework.

Remember the difference between Loss and Cost.

In [30]:
#------------------------------------------------------------------------------
#   SquareErrorLossFunction class
#------------------------------------------------------------------------------
class SELoss(Module):
    def __init__(self):
        super(SELoss, self).__init__()

    def forward(self, input: np.ndarray, target: np.ndarray) -> np.ndarray:
        # >>>>>>>>> add here
        m = input.shape[1]  # Number of samples
        self.diff = input - target
        loss = np.sum(np.square(self.diff))
        return loss
        # <<<<<<<<<

    def backward(self, input: np.ndarray, target: np.ndarray) -> np.ndarray:
        # >>>>>>>>> add here
        m = input.shape[1]  # Number of samples
        grad = 2 * (input - target)
        return grad
        # <<<<<<<<<

#------------------------------------------------------------------------------
#   BinaryCrossEntropyLossFunction class
#------------------------------------------------------------------------------
class BCELoss(Module):
    def __init__(self):
        super(BCELoss, self).__init__()

    def forward(self, input: np.ndarray, target: np.ndarray) -> np.ndarray:
        # >>>>>>>>> add here
        m = input.shape[1]  # Number of samples
        # Clip input to avoid log(0)
        input_clipped = np.clip(input, 1e-10, 1 - 1e-10)
        loss = -np.sum(target * np.log(input_clipped) + (1 - target) * np.log(1 - input_clipped)) / m
        return loss
        # <<<<<<<<<

    def backward(self, input: np.ndarray, target: np.ndarray) -> np.ndarray:
        # >>>>>>>>> add here
        m = input.shape[1]  # Number of samples
        # Clip input to avoid division by zero
        input_clipped = np.clip(input, 1e-10, 1 - 1e-10)
        grad = (input_clipped - target) / (input_clipped * (1 - input_clipped) * m)
        return -grad
        # <<<<<<<<<

## Model
As in previous task, use `Model` class to encapsulate all layers of your MLP and define backward pass.
Iterate over its modules stored in parameter OrderedDict `modules` -> `self.modules` in the correct order.

Use call `.add_module(...)` to add layers of your MLP (network). Define MLP that could classify data from Circles dataset `dataset_Circles(...)`.


In [5]:
#------------------------------------------------------------------------------
#   Model class
#------------------------------------------------------------------------------
from collections import OrderedDict

class Model(Module):
    def __init__(self):
        super(Model, self).__init__()
        self.modules = OrderedDict()

    def add_module(self, module, name):
        self.modules[name] = module

    def forward(self, input) -> np.ndarray:
        for name, module in self.modules.items():
            # print(f'Layer fw:{name}, a.shape = {input.shape} \n{input}')
            input = module(input)
            # print(f'z.shape = {input.shape} \n{input}')
        return input

    def backward(self, dz: np.ndarray):
        # >>>>>>>>> add here
        for name, module in reversed(self.modules.items()):
            dz = module.backward(dz)
        # <<<<<<<<<

## Main Processing Cell

 1. Initialize dataset (`dataset_Circles`). [x]
 2. Declare a simple model (at least 3 hidden layer). [x]
 3. Perform forward pass through the network. [x]
 4. Compute loss. []
 5. Backward prop loss. []
 6. Backward pass MLP. []
 7. Check your computation of gradients via [`gradient_check`](https://datascience-enthusiast.com/DL/Improving_DeepNeural_Networks_Gradient_Checking.html) []
 8. Start crying. []
 9. Repeat until correct ;) []
 10. ... (if error founds -> blame lecturer) []

In [6]:
from dataset import dataset_Circles
from utils import gradient_check

In [31]:
dataset_features_X, dataset_labels_Y = dataset_Circles(m=128, radius=0.7, noise=0.0)

mlp = Model()
mlp.add_module(Linear(2, 3), 'Dense_1')
mlp.add_module(Tanh(), 'Tanh_1')
mlp.add_module(Linear(3, 4), 'Dense_2')
mlp.add_module(Tanh(), 'Tanh_2')
mlp.add_module(Linear(4, 5), 'Dense_3')
mlp.add_module(Tanh(), 'Tanh_3')
mlp.add_module(Linear(5, 1), 'Dense_4_out')
mlp.add_module(Sigmoid(), 'Sigmoid')

predicted_Y_hat = mlp.forward(dataset_features_X) # Be careful with the shape - Loss vs Cost

In [32]:
###>>> start of solution

# TODO add loss function!!!
loss_fn = BCELoss()

###<<< end of solution

In [33]:
# Verify your solution!
gradient_check(mlp, loss_fn, dataset_features_X, dataset_labels_Y)

There is a mistake in the backward propagation! difference = 1.0
